# L1 与 L2 正则如何选择？

**面试回答主线：**L2 将所有权重平滑缩小，适合稳定处理共线特征；L1 通过零点尖角产生稀疏解，适合希望减少活跃特征或成本的场景。二者都必须在缩放后的训练数据和独立验证流程中选择。本实验手写 Ridge 梯度更新与 Lasso 软阈值近端更新。

## 真实案例

广告预估要预测每日转化数，特征是站内点击、与点击高度相关的曝光估计、优惠券次数和无关的天气编码。目标既要准确，也要减少线上读取无用特征的成本。

In [1]:
import numpy as np  # 导入 NumPy 以手写正则化优化。
np.set_printoptions(precision=3, suppress=True)  # 设置紧凑的数值格式。
campaign = np.array(['C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07', 'C08', 'C09', 'C10'])  # 构造广告活动编号。
click = np.array([2, 3, 5, 4, 7, 6, 8, 9, 10, 12], dtype=float)  # 记录站内点击次数。
impression_proxy = np.array([2.1, 2.9, 5.2, 3.8, 7.1, 5.9, 8.2, 8.8, 10.1, 11.7], dtype=float)  # 记录与点击高度相关的曝光代理特征。
coupon = np.array([0, 1, 0, 2, 1, 0, 2, 1, 0, 2], dtype=float)  # 记录发放优惠券次数。
weather_code = np.array([1, 0, 1, 0, 1, 0, 1, 0, 1, 0], dtype=float)  # 构造与目标无关的天气编码。
conversion = np.array([3, 5, 8, 8, 12, 11, 16, 17, 20, 25], dtype=float)  # 记录实际转化数。
feature = np.c_[click, impression_proxy, coupon, weather_code]  # 拼接四个广告特征。
print('活动 | 点击 | 曝光代理 | 券次数 | 天气码 | 转化')  # 输出业务样本表头。
for index in range(len(campaign)):  # 逐条展示广告活动样本。
    print(f'{campaign[index]} | {click[index]:4.0f} | {impression_proxy[index]:8.1f} | {coupon[index]:6.0f} | {weather_code[index]:6.0f} | {conversion[index]:4.0f}')  # 输出一条活动记录。

活动 | 点击 | 曝光代理 | 券次数 | 天气码 | 转化
C01 |    2 |      2.1 |      0 |      1 |    3
C02 |    3 |      2.9 |      1 |      0 |    5
C03 |    5 |      5.2 |      0 |      1 |    8
C04 |    4 |      3.8 |      2 |      0 |    8
C05 |    7 |      7.1 |      1 |      1 |   12
C06 |    6 |      5.9 |      0 |      0 |   11
C07 |    8 |      8.2 |      2 |      1 |   16
C08 |    9 |      8.8 |      1 |      0 |   17
C09 |   10 |     10.1 |      0 |      1 |   20
C10 |   12 |     11.7 |      2 |      0 |   25


## Baseline / 基线

基线只预测训练期平均转化数。它不读取任何投放信号，可衡量正则化模型是否至少学到了可泛化的关系。

In [2]:
train_index = np.arange(8)  # 将前八个活动作为训练窗口。
valid_index = np.arange(8, 10)  # 将最后两个活动作为未来验证窗口。
baseline_prediction = np.full(len(valid_index), conversion[train_index].mean())  # 用训练均值预测未来转化。
baseline_mse = float(np.mean((baseline_prediction - conversion[valid_index]) ** 2))  # 计算均值基线 MSE。
print('均值基线预测:', np.round(baseline_prediction, 2))  # 输出基线预测值。
print(f'均值基线 MSE={baseline_mse:.2f}')  # 输出基线指标。

均值基线预测: [10. 10.]
均值基线 MSE=162.50


In [3]:
mean = feature[train_index].mean(axis=0)  # 只在训练活动上拟合特征均值。
std = feature[train_index].std(axis=0) + 1e-6  # 只在训练活动上拟合标准差并避免除零。
x_train = (feature[train_index] - mean) / std  # 标准化训练特征让惩罚具有可比较含义。
x_valid = (feature[valid_index] - mean) / std  # 使用训练统计量标准化未来活动。
y_train = conversion[train_index] - conversion[train_index].mean()  # 中心化训练目标以单独处理截距。
y_mean = conversion[train_index].mean()  # 保存部署时应恢复的目标均值。
def soft_threshold(value, threshold):  # 定义 L1 近端更新所需的软阈值算子。
    return np.sign(value) * np.maximum(np.abs(value) - threshold, 0.0)  # 将靠近零的权重直接压成零。
print('特征相关矩阵:', np.round(np.corrcoef(x_train.T), 2))  # 输出点击与曝光代理的共线性中间量。

特征相关矩阵: [[ 1.    1.    0.31  0.  ]
 [ 1.    1.    0.3   0.07]
 [ 0.31  0.3   1.   -0.16]
 [ 0.    0.07 -0.16  1.  ]]


In [4]:
def fit_ridge(x, y, l2_strength):  # 定义 Ridge 的批量梯度下降函数。
    weight = np.zeros(x.shape[1])  # 初始化 Ridge 权重。
    for step in range(800):  # 重复执行足够多次的平滑更新。
        gradient = x.T @ (x @ weight - y) / len(x) + l2_strength * weight  # 计算 MSE 加 L2 的梯度。
        weight -= 0.08 * gradient  # 沿负梯度更新所有权重。
    return weight  # 返回训练后的 Ridge 权重。
def fit_lasso(x, y, l1_strength, steps=800, step_size=0.08):  # 定义可控制步数和步长的 Lasso 近端梯度函数。
    weight = np.zeros(x.shape[1])  # 初始化 Lasso 权重。
    for step in range(steps):  # 重复执行指定次数的梯度和软阈值更新。
        smooth_gradient = x.T @ (x @ weight - y) / len(x)  # 计算未含 L1 的平滑 MSE 梯度。
        weight = soft_threshold(weight - step_size * smooth_gradient, step_size * l1_strength)  # 对梯度步结果施加 L1 软阈值。
    return weight  # 返回稀疏的 Lasso 权重。
ridge_weight = fit_ridge(x_train, y_train, 0.25)  # 拟合 L2 正则模型。
lasso_weight = fit_lasso(x_train, y_train, 0.45)  # 拟合 L1 正则模型。
print('Ridge 权重:', np.round(ridge_weight, 3))  # 展示 L2 对全部特征的平滑收缩。
print('Lasso 权重:', np.round(lasso_weight, 3))  # 展示 L1 将部分特征压零的中间结果。

Ridge 权重: [ 1.97   1.954  0.543 -0.233]
Lasso 权重: [ 2.738  1.32   0.215 -0.   ]


In [5]:
ridge_prediction = x_valid @ ridge_weight + y_mean  # 用 Ridge 权重预测未来转化数。
lasso_prediction = x_valid @ lasso_weight + y_mean  # 用 Lasso 权重预测未来转化数。
ridge_mse = float(np.mean((ridge_prediction - conversion[valid_index]) ** 2))  # 计算 Ridge 验证 MSE。
lasso_mse = float(np.mean((lasso_prediction - conversion[valid_index]) ** 2))  # 计算 Lasso 验证 MSE。
ridge_active = int(np.sum(np.abs(ridge_weight) > 1e-3))  # 统计 Ridge 的活跃特征数量。
lasso_active = int(np.sum(np.abs(lasso_weight) > 1e-3))  # 统计 Lasso 的活跃特征数量。
print('方案   | 未来MSE | 活跃特征数 | 权重')  # 输出结果表头。
print(f'Ridge  | {ridge_mse:7.2f} | {ridge_active:10d} | {np.round(ridge_weight, 2)}')  # 输出 L2 结果。
print(f'Lasso  | {lasso_mse:7.2f} | {lasso_active:10d} | {np.round(lasso_weight, 2)}')  # 输出 L1 结果。

方案   | 未来MSE | 活跃特征数 | 权重
Ridge  |    9.55 |          4 | [ 1.97  1.95  0.54 -0.23]
Lasso  |    8.11 |          3 | [ 2.74  1.32  0.22 -0.  ]


## 结果解读

点击与曝光代理高度相关，L2 往往保留两者并共同收缩，参数更平滑；L1 更可能只保留一个或把无关天气码压为零，从而减少特征读取。是否选择 L1 不能只看 MSE，还要看不同训练窗口下被选特征是否稳定。

In [6]:
feature_name = np.array(['点击', '曝光代理', '优惠券', '天气码'])  # 构造可解释的特征名称数组。
print('特征 | Ridge权重 | Lasso权重')  # 输出逐特征权重表头。
for index in range(len(feature_name)):  # 逐项解释正则化后的参数。
    print(f'{feature_name[index]:5s} | {ridge_weight[index]:9.3f} | {lasso_weight[index]:9.3f}')  # 输出同一特征在两种惩罚下的权重。
print('教学结论：零权重是优化结果，不是因果结论；相关特征下 L1 的选择可能随样本变化。')  # 说明稀疏性不能被过度解释。

特征 | Ridge权重 | Lasso权重
点击    |     1.970 |     2.738
曝光代理  |     1.954 |     1.320
优惠券   |     0.543 |     0.215
天气码   |    -0.233 |    -0.000
教学结论：零权重是优化结果，不是因果结论；相关特征下 L1 的选择可能随样本变化。


## 失败案例与修复

错误做法是对未缩放特征施加相同 L1/L2 系数：大单位变量受到的惩罚含义会改变。下面把点击放大 100 倍，观察同一 L1 强度的选择扭曲；修复是训练集标准化并将统计量版本化。

In [7]:
unscaled_feature = feature[train_index].copy()  # 复制训练特征以构造未缩放反例。
unscaled_feature[:, 0] *= 100.0  # 人为把点击特征单位放大一百倍。
bad_lasso_weight = fit_lasso(unscaled_feature, y_train, 0.45, steps=4, step_size=0.00002)  # 用有限轮次但偏大的步长构造数值仍有限的发散反例。
bad_lasso_norm = float(np.linalg.norm(bad_lasso_weight))  # 计算未缩放反例的参数范数。
print('失败：放大点击后的有限权重/范数:', np.round(bad_lasso_weight, 4), round(bad_lasso_norm, 2))  # 展示单位改变和过大步长造成的可解释放大。
print('修复：标准化后的 Lasso 权重:', np.round(lasso_weight, 4))  # 展示可比较尺度上的结果。
print('生产差距：需做嵌套验证选 lambda、跟踪特征成本与稳定性，并避免将系数当作因果效应。')  # 说明生产选择正则的额外要求。

失败：放大点击后的有限权重/范数: [-4.091 -0.041 -0.006 -0.003] 4.09
修复：标准化后的 Lasso 权重: [ 2.737  1.32   0.215 -0.   ]
生产差距：需做嵌套验证选 lambda、跟踪特征成本与稳定性，并避免将系数当作因果效应。


In [8]:
assert len(campaign) >= 5  # 保护案例包含至少五条广告活动。
assert lasso_active <= ridge_active  # 保护 L1 在该实验中不比 L2 更稠密。
assert min(ridge_mse, lasso_mse) < baseline_mse  # 保护至少一个正则模型优于均值基线。
assert not np.allclose(bad_lasso_weight, lasso_weight)  # 保护未缩放会改变正则化结果这一失败现象。